# Uzbek NER — mmBERT на Kaggle

Ноутбук обучает `jhu-clsp/mmBERT-base` через Hugging Face `Trainer`, сохраняет исправленную логику перекрывающихся окон и считает конкурсные exact-span precision/recall/F1. Результаты записываются в `/kaggle/working`.

Перед запуском выберите GPU в **Settings → Accelerator**. Если в образе Kaggle не хватает зависимостей, раскомментируйте следующую ячейку и после установки перезапустите сессию.

In [ ]:
# %pip install -q "transformers>=4.48" "tokenizers>=0.20" "accelerate>=1.0" "tqdm>=4.66"

## 1. Конфигурация

Для первых двух экспериментов достаточно менять только `DATA_VARIANT`. В режиме `augmented` берётся нормализованный train и добавляются файлы аугментаций. `MAX_AUGMENTED_RECORDS` ограничивает их количество, чтобы синтетика не задавила оригинальные данные; `None` означает взять всё.

In [ ]:
from pathlib import Path
import json
import random
import sys

# original: train_original.jsonl + dev_original.jsonl
# normalized: train.jsonl + dev.jsonl
# augmented: train.jsonl + аугментации; validation остаётся dev.jsonl
DATA_VARIANT = "original"

MODEL_NAME = "jhu-clsp/mmBERT-base"
MAX_LENGTH = 512
STRIDE = 128
EPOCHS = 10
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
EARLY_STOPPING_PATIENCE = 2
SEED = 42

# Для режима augmented. Равномерная случайная подвыборка из общего пула.
MAX_AUGMENTED_RECORDS = 13_000

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/ner_runs")

# В Kaggle имя dataset может меняться, поэтому ищем две реально нужные папки.
COMMON_PATH = next(INPUT_ROOT.rglob("baseline/common.py"))
PROJECT_DIR = COMMON_PATH.parent.parent
DATA_DIR = next(
    path.parent
    for path in INPUT_ROOT.rglob("train_original.jsonl")
    if (path.parent / "dev_original.jsonl").exists()
)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

model_slug = MODEL_NAME.rsplit("/", 1)[-1]
RUN_DIR = OUTPUT_ROOT / f"{DATA_VARIANT}_{model_slug}"
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
MODEL_DIR = RUN_DIR / "model"
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR:   ", DATA_DIR)
print("RUN_DIR:    ", RUN_DIR)

## 2. Данные

`original` и `normalized` отличаются только представлением апострофов. Нормализация сохраняет длину строки, поэтому символьные координаты сущностей не сдвигаются. В режиме `augmented` к нормализованному train добавляются case, hard-negative, morphology и script-swap примеры.

In [ ]:
from baseline.common import (
    TAGS,
    TAG_TO_ID,
    TokenizedNerDataset,
    align_labels,
    load_fast_tokenizer,
    read_records,
    set_seed,
    tokenize_windows,
    validate_window,
)
from baseline.predict import _build_windows, _decode_records, _predict_token_scores, _write_jsonl
from scripts.evaluate import calculate_metrics, print_metrics

ID_TO_TAG = {index: tag for index, tag in enumerate(TAGS)}
set_seed(SEED, seed_cuda=True)

if DATA_VARIANT == "original":
    train_path = DATA_DIR / "train_original.jsonl"
    dev_path = DATA_DIR / "dev_original.jsonl"
    augmentation_paths = []
elif DATA_VARIANT == "normalized":
    train_path = DATA_DIR / "train.jsonl"
    dev_path = DATA_DIR / "dev.jsonl"
    augmentation_paths = []
elif DATA_VARIANT == "augmented":
    train_path = DATA_DIR / "train.jsonl"
    dev_path = DATA_DIR / "dev.jsonl"
    augmentation_paths = [
        DATA_DIR / "case.jsonl",
        DATA_DIR / "hard_negatives.jsonl",
        DATA_DIR / "morphology.jsonl",
        *sorted(DATA_DIR.glob("part-*.jsonl")),
    ]
else:
    raise ValueError("DATA_VARIANT должен быть original, normalized или augmented")

train_records = read_records(train_path, require_entities=True)
dev_records = read_records(dev_path, require_entities=True)

if augmentation_paths:
    augmented_records = []
    for path in augmentation_paths:
        if path.exists():
            augmented_records.extend(read_records(path, require_entities=True))

    random.Random(SEED).shuffle(augmented_records)
    if MAX_AUGMENTED_RECORDS is not None:
        augmented_records = augmented_records[:MAX_AUGMENTED_RECORDS]
    train_records.extend(augmented_records)
    print(f"Добавлено аугментаций: {len(augmented_records):,}")

print(f"Train: {len(train_records):,} документов ({train_path.name})")
print(f"Dev:   {len(dev_records):,} документов ({dev_path.name})")

## 3. Оконная токенизация mmBERT

Fast-tokenizer mmBERT добавляет два служебных токена: начальный `<bos>` и конечный `<eos>`. Функция `tokenize_windows` из `baseline/common.py` поддерживает эту схему. Локальный адаптер ниже дополнительно убирает внешние пробелы из token offsets: mmBERT включает ведущий пробел в координаты токенов с маркером `▁`, что без адаптера нарушает exact-span BIO-разметку.

In [ ]:
import baseline.common as common_module
import baseline.predict as predict_module


def _trim_whitespace_offsets(text, offsets):
    trimmed = []
    for start, end in offsets:
        start = int(start)
        end = int(end)
        while start < end and text[start].isspace():
            start += 1
        while end > start and text[end - 1].isspace():
            end -= 1
        trimmed.append((start, end))
    return trimmed


# Сохраняем исходную реализацию: ячейку можно безопасно перезапускать.
if not hasattr(common_module, "_untrimmed_tokenize_windows"):
    common_module._untrimmed_tokenize_windows = common_module.tokenize_windows


def tokenize_windows_with_trimmed_offsets(
    tokenizer,
    text,
    *,
    max_length,
    stride,
):
    windows = common_module._untrimmed_tokenize_windows(
        tokenizer,
        text,
        max_length=max_length,
        stride=stride,
    )
    return [
        (feature, _trim_whitespace_offsets(text, offsets))
        for feature, offsets in windows
    ]


# Dataset, notebook и baseline.predict должны использовать одни offsets.
common_module.tokenize_windows = tokenize_windows_with_trimmed_offsets
predict_module.tokenize_windows = tokenize_windows_with_trimmed_offsets
tokenize_windows = tokenize_windows_with_trimmed_offsets

print("mmBERT whitespace-offset patch applied")

tokenizer = load_fast_tokenizer(MODEL_NAME)
validate_window(tokenizer, MAX_LENGTH, STRIDE)

offset_probe_text = "Salom, Toshkent shahridagi OpenAI kompaniyasi."
_, offset_probe = tokenize_windows(
    tokenizer,
    offset_probe_text,
    max_length=MAX_LENGTH,
    stride=STRIDE,
)[0]
assert all(
    start == end
    or (
        not offset_probe_text[start].isspace()
        and not offset_probe_text[end - 1].isspace()
    )
    for start, end in offset_probe
)
print("Whitespace-free token offsets verified")

print(
    "Special tokens:",
    tokenizer.num_special_tokens_to_add(pair=False),
    tokenizer.bos_token,
    tokenizer.eos_token,
)

train_dataset = TokenizedNerDataset(
    train_records,
    tokenizer,
    max_length=MAX_LENGTH,
    stride=STRIDE,
    description="Train tokenization",
)

dev_windows = _build_windows(
    dev_records,
    tokenizer,
    max_length=MAX_LENGTH,
    stride=STRIDE,
)

# Trainer должен получить labels, а offsets остаются отдельно для exact-span decoding.
eval_dataset = []
for record_index, feature, offsets in dev_windows:
    labels = align_labels(offsets, dev_records[record_index]["entities"])
    eval_dataset.append({**feature, "labels": labels})

# Проверяем исправление прежней ошибки: последний токен длинного текста не потерян.
longest_record = max(dev_records, key=lambda record: len(record["text"]))
longest_windows = tokenize_windows(
    tokenizer,
    longest_record["text"],
    max_length=MAX_LENGTH,
    stride=STRIDE,
)
full_offsets = _trim_whitespace_offsets(
    longest_record["text"],
    tokenizer(
        longest_record["text"],
        add_special_tokens=False,
        truncation=False,
        return_offsets_mapping=True,
    )["offset_mapping"],
)
last_full_end = max((end for start, end in full_offsets if start != end), default=0)
last_window_end = max(
    end
    for _, offsets in longest_windows
    for start, end in offsets
    if start != end
)
assert last_window_end == last_full_end

print(f"Train windows: {len(train_dataset):,}")
print(f"Dev windows:   {len(eval_dataset):,}")
print(f"Longest dev document: {len(longest_windows)} windows, covered through char {last_window_end}")

## 4. Конкурсная метрика

Обычная token-level accuracy здесь вводит в заблуждение из-за большого числа токенов `O` (не сущность). Поэтому каждая validation эпоха оценивается по полному совпадению класса и границ сущности: `label + start + end`. Лучший checkpoint выбирается по micro-F1; precision также выводится отдельно.

In [ ]:
import numpy as np
import torch


def gold_for_metrics(records):
    return {
        record["hash"]: {
            "entities": {
                (entity["label"], entity["start"], entity["end"])
                for entity in record["entities"]
            }
        }
        for record in records
    }


def predictions_for_metrics(predictions):
    return {
        record["hash"]: {
            (entity["label"], entity["start"], entity["end"])
            for entity in record["entities"]
        }
        for record in predictions
    }


DEV_GOLD = gold_for_metrics(dev_records)


def decode_window_logits(raw_logits):
    logits = np.asarray(raw_logits)
    scores = [{} for _ in dev_records]

    for row_index, (record_index, _, offsets) in enumerate(dev_windows):
        probabilities = torch.softmax(
            torch.as_tensor(logits[row_index, :len(offsets)]).float(),
            dim=-1,
        )
        record_scores = scores[record_index]
        for token_index, (start, end) in enumerate(offsets):
            if start == end:
                continue
            key = (start, end)
            score = probabilities[token_index]
            if key in record_scores:
                previous, count = record_scores[key]
                record_scores[key] = (previous + score, count + 1)
            else:
                record_scores[key] = (score.clone(), 1)

    return _decode_records(dev_records, scores, ID_TO_TAG)


def compute_metrics(eval_prediction):
    raw_logits = eval_prediction.predictions
    if isinstance(raw_logits, tuple):
        raw_logits = raw_logits[0]

    predictions = decode_window_logits(raw_logits)
    metrics = calculate_metrics(DEV_GOLD, predictions_for_metrics(predictions))

    result = {
        "micro_precision": metrics["micro"]["precision"],
        "micro_recall": metrics["micro"]["recall"],
        "micro_f1": metrics["micro"]["f1"],
    }
    for label, values in metrics["by_label"].items():
        result[f"{label}_f1"] = values["f1"]
    return result

## 5. Модель и обучение

`Trainer` сам выполняет mixed precision, gradient accumulation, AdamW, linear scheduler, validation, раннюю остановку и загрузку лучшего checkpoint. Ручной цикл обучения здесь не нужен.

In [ ]:
from transformers import (
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

if not torch.cuda.is_available():
    raise RuntimeError("Включите GPU в Kaggle: Settings → Accelerator")

bf16_available = torch.cuda.is_bf16_supported()

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(TAGS),
    id2label=ID_TO_TAG,
    label2id=TAG_TO_ID,
)

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="linear",
    max_grad_norm=1.0,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,
    save_total_limit=2,
    #bf16=bf16_available,
    fp16=True, #not bf16_available,
    eval_accumulation_steps=4,
    dataloader_num_workers=2,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

train_result = trainer.train()
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best micro-F1:  ", trainer.state.best_metric)

## 6. Сохранение лучшей модели и финальная оценка

После `train()` в `trainer.model` уже загружен лучший checkpoint, а не веса последней эпохи. Сохраняются модель, tokenizer, параметры окон, validation-предсказания и полный отчёт метрик.

In [ ]:
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

baseline_config = {
    "model_name": MODEL_NAME,
    "data_variant": DATA_VARIANT,
    "max_length": MAX_LENGTH,
    "stride": STRIDE,
}
(MODEL_DIR / "baseline_config.json").write_text(
    json.dumps(baseline_config, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

device = next(trainer.model.parameters()).device
dev_scores = _predict_token_scores(
    trainer.model,
    tokenizer,
    dev_windows,
    len(dev_records),
    batch_size=EVAL_BATCH_SIZE,
    device=device,
)
dev_predictions = _decode_records(dev_records, dev_scores, ID_TO_TAG)
dev_metrics = calculate_metrics(DEV_GOLD, predictions_for_metrics(dev_predictions))

In [ ]:
# Запись результатов вынесена отдельно, чтобы эту ячейку можно было повторить без обучения.
predictions_path = RUN_DIR / "dev_predictions.jsonl"
metrics_path = RUN_DIR / "dev_metrics.json"

_write_jsonl(predictions_path, dev_predictions)
metrics_path.write_text(
    json.dumps(dev_metrics, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

print_metrics(dev_metrics)
print("\nModel:      ", MODEL_DIR)
print("Predictions:", predictions_path)
print("Metrics:    ", metrics_path)

## Следующие прогоны

1. Запустите `DATA_VARIANT = "original"` и запишите micro-precision/micro-F1.
2. Перезапустите kernel, выберите `normalized` и сравните метрики.
3. Затем выберите `augmented`. Начните с лимита 13 000; после этого отдельно проверяйте другие размеры пула.

Каждый вариант сохраняется в свою папку внутри `/kaggle/working/ner_runs`, поэтому результаты друг друга не перезаписывают.